# Baseline Predictive Maintenance Model

This notebook records the first baseline result for the predictive maintenance classification project,

The goal is to evaluate a Logistic Regression baseline trained on the AI4I 2020 Predictive Maintenance dataset and understand its behavior.

In [1]:
import numpy as np
import pandas as pd

from predictive_maintenance_ml.experiment import BaselineExperiment
from predictive_maintenance_ml.evaluation import ClassificationEvaluator

In [2]:
experiment = BaselineExperiment()
results = experiment.run()

y_test = results["y_test"]
y_pred = results["y_pred"]
y_proba = results["y_proba"]

In [3]:
evaluator = ClassificationEvaluator()

metrics = evaluator.evaluate(
    y_true=y_test,
    y_pred=y_pred,
    y_proba=y_proba,
)

cm = metrics.pop('confusion_matrix')
cl_report = metrics.pop('classification_report')

metrics_df = pd.DataFrame({'Metrics':metrics})

In [4]:
print(cm)
print(cl_report)
metrics_df.head()

[[1593  339]
 [  12   56]]
              precision    recall  f1-score   support

           0       0.99      0.82      0.90      1932
           1       0.14      0.82      0.24        68

    accuracy                           0.82      2000
   macro avg       0.57      0.82      0.57      2000
weighted avg       0.96      0.82      0.88      2000



,Metrics
accuracy,0.824500
precision,0.141772
recall,0.823529
f1,0.241901
roc_auc,0.906954


In [5]:
print(metrics)

{'accuracy': 0.8245, 'precision': 0.14177215189873418, 'recall': 0.8235294117647058, 'f1': 0.24190064794816415, 'roc_auc': 0.9069540859822189}


## Baseline result

The Logistic Regression baseline produced the following result on the test set:

- Accuracy: 0.8245
- Precision: 0.1418
- Recall: 0.8235
- F1-score: 0.2419
- ROC-AUC: 0.9070

The confusion matrix was:

|                | Predicted normal | Predicted failure |
|----------------|------------------|-------------------|
| True normal    | 1593             | 339               |
| True failure   | 12               | 56                |

## Interpretation

The model detects most failure cases: it correctly identifies 56 out of 68 failures in the test set. This gives a recall of approximately 82.35% for the failure class.

However, the model also generates many false alarms: 339 normal samples are incorrectly classified as failures. As a result, precision for the failure class is only approximately 14.18%.

This means that the model is aggressive in predicting failures. It is useful if the priority is to avoid missing failures, but it would likely create too many unnecessary inspections in a real maintenance setting.

## Why accuracy is misleading

The dataset is highly imbalanced. In the test set, there are 1932 normal samples and only 68 failure samples.

A naive model that always predicts "normal" would achieve high accuracy, but it would detect no failures. For this reason, accuracy is not the main metric for this problem.

The more relevant metrics are:

- recall for the failure class,
- precision for the failure class,
- F1-score,
- confusion matrix,
- ROC-AUC,
- and later precision-recall analysis.

In [6]:
print(y_test.value_counts(normalize=True))
print(y_test.value_counts())

Machine failure
0    0.966
1    0.034
Name: proportion, dtype: float64
Machine failure
0    1932
1      68
Name: count, dtype: int64


## Threshold sweep

The baseline model has a good ROC-AUC score, which suggests that it can rank risky samples well. However, the default decision threshold produces many false positives.

The next step is to evaluate different probability thresholds and study the tradeoff between precision and recall
A higher threshold should reduce false positives and improve precision, but it may also increase false negatives and reduce recall.

From the sweep, we can see that the default threshold is not necessarily optimal. Adjusting the decision threshold allows the model to trade recall against precision depending on the operational cost of missed failures versus false alarms. The threshold of `0.8` gives the higher `F1` score, in fact it reduces false alarm massively with respect to the default threshold of `0.5`, but at the same time, it misses more failures. This is the main tradeoff to consider when choosing the threshold.

In [8]:
threshold_results = evaluator.evaluate_thresholds(
    y_true=y_test,
    y_proba=y_proba,
)

threshold_results.sort_values("f1", ascending=False).reset_index(drop=True)

,threshold,accuracy,precision,recall,f1,true_negatives,false_positives,false_negatives,true_positives
0,0.8,0.9470,0.338983,0.588235,0.430108,1854,78,28,40
1,0.9,0.9625,0.433962,0.338235,0.380165,1902,30,45,23
2,0.7,0.9160,0.250000,0.735294,0.373134,1782,150,18,50
3,0.6,0.8760,0.183099,0.764706,0.295455,1700,232,16,52
4,0.5,0.8245,0.141772,0.823529,0.241901,1593,339,12,56
5,0.4,0.7620,0.115094,0.897059,0.204013,1463,469,7,61
6,0.3,0.6865,0.091971,0.926471,0.167331,1310,622,5,63
7,0.2,0.5825,0.072464,0.955882,0.134715,1100,832,3,65
8,0.1,0.4025,0.053132,0.985294,0.100828,738,1194,1,67
